# Experiment 0001 — Reproduce & measure catastrophic forgetting

This notebook **is** the experiment, not a sandbox. Read
[`../../docs/methodology.md`](../../docs/methodology.md) and
[`README.md`](README.md) first. It does exactly one thing:

> Train a small net on the 5 Split-MNIST tasks **one after another**
> (sequential), and on **all of them at once** (joint = the no-forgetting
> reference). Measure how much task-1 accuracy collapses, and how much more
> the joint baseline costs in wall-clock.

It tests charter hypotheses **H1** (forgetting drop ≥ 40 pts) and **H2**
(joint costs ≥ 3× sequential). No method here — you cannot claim a fix for
a thing you haven't measured.

Setup: see [`../../docs/setup.md`](../../docs/setup.md) —
`uv sync --extra experiments` then open this with the `.venv` kernel.

In [1]:
import time, json, platform
import torch
from torch.utils.data import DataLoader
import avalanche, researchkit

print("torch     ", torch.__version__)
print("avalanche ", avalanche.__version__, "  # API shifts between versions; if an import below fails, check this + the Avalanche docs")
print("researchkit", researchkit.__version__)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device    ", device)

torch      2.12.0
avalanche  0.6.0   # API shifts between versions; if an import below fails, check this + the Avalanche docs
researchkit 0.1.0
device     mps


/Users/chintu/Research_Papers/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## ⛔ PRE-REGISTRATION GATE — fill this before running anything below

Methodology rule: predict the result *before* you see it. Being wrong here is
the single most useful output of this notebook — it calibrates how badly you
currently understand forgetting. The next cell **raises an error until you
replace the `None`s**. Do not peek ahead first.

In [2]:
# Replace each None with your honest guess. No looking ahead.
PRED_TASK1_ACC_AFTER_T1 = 98.0   # your guess: task-1 acc right after learning task 1
PRED_TASK1_ACC_AFTER_T5 = 30.0   # your guess: task-1 acc after learning all 5 sequentially
PRED_JOINT_OVER_SEQ_COST = 3.0   # your guess: joint ÷ sequential wall-clock
assert None not in (PRED_TASK1_ACC_AFTER_T1, PRED_TASK1_ACC_AFTER_T5, PRED_JOINT_OVER_SEQ_COST), (
    "Fill in your three predictions before running the experiment.")
print("Pre-registered:", PRED_TASK1_ACC_AFTER_T1, PRED_TASK1_ACC_AFTER_T5, PRED_JOINT_OVER_SEQ_COST)

Pre-registered: 98.0 30.0 3.0


In [3]:
SEED = 0
researchkit.set_global_seed(SEED)
ENV = researchkit.capture_env()
print(json.dumps(ENV, indent=2))  # paste this into README 'Setup' when done

{
  "git_commit": "94f6509-dirty",
  "python": "3.11.13",
  "platform": "macOS-26.2-arm64-arm-64bit",
  "argv": [
    "/Users/chintu/Research_Papers/.venv/lib/python3.11/site-packages/ipykernel_launcher.py",
    "-f",
    "/Users/chintu/Library/Jupyter/runtime/kernel-5f199218-dadd-4944-9449-d62e8129fb89.json"
  ]
}


## Build the benchmark

**Split-MNIST, class-incremental:** task 1 = digits {0,1}, task 2 = {2,3}, …,
task 5 = {8,9}. Class-incremental is the *hardest* of the three settings in
van de Ven & Tolias 2019 (paper E in the reading list) — the model never sees
a task id at test time, so it must keep all 10 classes straight. We pick the
hard setting deliberately: if forgetting is mild even here, the benchmark is
too easy and H1 should be rejected.

In [4]:
from avalanche.benchmarks.classic import SplitMNIST

bm = SplitMNIST(n_experiences=5, seed=SEED, return_task_id=False)
train_exps = list(bm.train_stream)
test_exps  = list(bm.test_stream)
N_TASKS = len(train_exps)
print(N_TASKS, "tasks; classes per task:",
      [sorted(set(int(y) for *_, y in [e.dataset[i] for i in range(0, len(e.dataset), max(1, len(e.dataset)//200))])) for e in train_exps])

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9.91M/9.91M [00:05<00:00, 1.73MB/s]
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 28.9k/28.9k [00:00<00:00, 132kB/s]
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1.65M/1.65M [00:02<00:00, 600kB/s]
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4.54k/4.54k [00:00<00:00, 1.50MB/s]


5 tasks; classes per task: [[0], [0], [0], [0], [0]]


In [5]:
from avalanche.models import SimpleMLP

def fresh_model():
    researchkit.set_global_seed(SEED)        # same init for both regimes
    return SimpleMLP(num_classes=10, hidden_size=256).to(device)

@torch.no_grad()
def accuracy(model, dataset):
    """Plain top-1 accuracy. Written by hand on purpose: you should see
    exactly how the number is computed, not trust a metric dict."""
    model.eval()
    loader = DataLoader(dataset, batch_size=512)
    correct = total = 0
    for x, y, *_ in loader:                   # Avalanche yields (x, y, task_id)
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        total   += y.numel()
    return 100.0 * correct / total

## The one variable: training order

Everything (architecture, optimizer, epochs, seed, data) is held fixed. The
*only* thing that differs between the two runs below is **sequential vs
joint** training. That is what makes any difference attributable.

We record an accuracy matrix `R` where `R[i][j]` = accuracy on task *j* after
finishing training task *i*. From it (GEM defs, paper #2):
- **Average Accuracy** = mean of the last row.
- **Backward Transfer (BWT)** = mean over j<last of `R[last][j] - R[j][j]`
  (negative = forgetting).
- **Task-1 drop** = `R[0][0] - R[last][0]` — the headline H1 number.

In [6]:
from avalanche.training.supervised import Naive

EPOCHS, MB = 1, 128                # tiny on purpose — fast loop, enough signal
model = fresh_model()
opt = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
naive = Naive(model, opt, torch.nn.CrossEntropyLoss(),
              train_mb_size=MB, train_epochs=EPOCHS, eval_mb_size=512,
              device=device)

R = [[0.0] * N_TASKS for _ in range(N_TASKS)]
t0 = time.perf_counter()
for i, exp in enumerate(train_exps):
    naive.train(exp)
    for j, te in enumerate(test_exps):
        R[i][j] = accuracy(model, te.dataset)
    print(f"after task {i+1}: task1={R[i][0]:.1f}  thisTask={R[i][i]:.1f}")
seq_seconds = time.perf_counter() - t0
print(f"sequential wall-clock: {seq_seconds:.1f}s")

/Users/chintu/Research_Papers/.venv/lib/python3.11/site-packages/avalanche/training/templates/base.py:468: PositionalArgumentsDeprecatedWarning: Avalanche is transitioning to strategy constructors that accept named (keyword) arguments only. This is done to ensure that there is no confusion regarding the meaning of each argument (strategies can have many arguments). Your are passing 3 positional arguments to the Naive.__init__ method. Consider passing them as names arguments. The ability to pass positional arguments will be removed in the future.
  warnings.warn(error_str, category=PositionalArgumentsDeprecatedWarning)


-- >> Start of training phase << --
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 99/99 [00:02<00:00, 38.61it/s]
Epoch 0 ended.
	Loss_Epoch/train_phase/train_stream/Task000 = 0.0352
	Top1_Acc_Epoch/train_phase/train_stream/Task000 = 0.9762
-- >> End of training phase << --
after task 1: task1=0.0  thisTask=0.0
-- >> Start of training phase << --
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 92/92 [00:00<00:00, 92.16it/s]
Epoch 0 ended.
	Loss_Epoch/train_phase/train_stream/Task000 = 0.0000
	Top1_Acc_Epoch/train_phase/train_stream/Task000 = 0.9843
-- >> End of training phase << --
after task 2: task1=0.0  thisTask=0.0
-- >> Start of training phase << --
100%|███████████████████████████████████████████████████████████████████████

In [7]:
# Baseline / no-forgetting reference: train once on ALL tasks jointly.
from avalanche.training.supervised import JointTraining

jmodel = fresh_model()
jopt = torch.optim.SGD(jmodel.parameters(), lr=0.05, momentum=0.9)
joint = JointTraining(jmodel, jopt, torch.nn.CrossEntropyLoss(),
                      train_mb_size=MB, train_epochs=EPOCHS, eval_mb_size=512,
                      device=device)
t0 = time.perf_counter()
joint.train(bm.train_stream)        # JointTraining concatenates all experiences
joint_seconds = time.perf_counter() - t0
joint_acc = [accuracy(jmodel, te.dataset) for te in test_exps]
print(f"joint wall-clock: {joint_seconds:.1f}s  per-task acc: "
      + " ".join(f"{a:.1f}" for a in joint_acc))

/Users/chintu/Research_Papers/.venv/lib/python3.11/site-packages/avalanche/training/templates/base.py:468: PositionalArgumentsDeprecatedWarning: Avalanche is transitioning to strategy constructors that accept named (keyword) arguments only. This is done to ensure that there is no confusion regarding the meaning of each argument (strategies can have many arguments). Your are passing 3 positional arguments to the JointTraining.__init__ method. Consider passing them as names arguments. The ability to pass positional arguments will be removed in the future.
  warnings.warn(error_str, category=PositionalArgumentsDeprecatedWarning)


-- >> Start of training phase << --
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 469/469 [00:03<00:00, 130.52it/s]
Epoch 0 ended.
	Loss_Epoch/train_phase/train_stream/Task000 = 0.0068
	Top1_Acc_Epoch/train_phase/train_stream/Task000 = 0.9825
-- >> End of training phase << --
joint wall-clock: 3.6s  per-task acc: 0.0 0.0 0.0 50.2 0.0


In [8]:
last = N_TASKS - 1
avg_acc_seq = sum(R[last]) / N_TASKS
bwt = sum(R[last][j] - R[j][j] for j in range(last)) / last
task1_drop = R[0][0] - R[last][0]
cost_mult = joint_seconds / seq_seconds

print("=== RESULTS (raw) ===")
print(f"task-1 acc after T1 : {R[0][0]:.1f}%   (predicted {PRED_TASK1_ACC_AFTER_T1})")
print(f"task-1 acc after T5 : {R[last][0]:.1f}%   (predicted {PRED_TASK1_ACC_AFTER_T5})")
print(f"task-1 DROP         : {task1_drop:.1f} pts   [H1 needs >= 40]")
print(f"avg acc (sequential): {avg_acc_seq:.1f}%")
print(f"backward transfer   : {bwt:.1f}  (negative = forgetting)")
print(f"avg acc (joint ref) : {sum(joint_acc)/N_TASKS:.1f}%")
print(f"cost multiple       : {cost_mult:.2f}x   (predicted {PRED_JOINT_OVER_SEQ_COST})  [H2 needs >= 3]")
print("\nAccuracy matrix R[i][j] (row = after task i+1):")
for row in R:
    print("  " + " ".join(f"{v:5.1f}" for v in row))

=== RESULTS (raw) ===
task-1 acc after T1 : 0.0%   (predicted 98.0)
task-1 acc after T5 : 0.0%   (predicted 30.0)
task-1 DROP         : 0.0 pts   [H1 needs >= 40]
avg acc (sequential): 10.0%
backward transfer   : 0.0  (negative = forgetting)
avg acc (joint ref) : 10.0%
cost multiple       : 0.45x   (predicted 3.0)  [H2 needs >= 3]

Accuracy matrix R[i][j] (row = after task i+1):
    0.0   0.0   0.0  50.2   0.0
    0.0   0.0   0.0  50.2   0.0
    0.0   0.0   0.0  50.2   0.0
    0.0   0.0   0.0  50.2   0.0
    0.0   0.0   0.0  50.2   0.0


## Now YOU conclude (the notebook deliberately does not)

Open [`README.md`](README.md) and fill in, in your own words:

1. **H1 supported or rejected?** Is `task1_drop >= 40`? If forgetting is mild,
   the benchmark is too easy — say so, that's a real finding.
2. **H2 supported or rejected?** Is `cost_mult >= 3`? Note: with 1 epoch and 5
   tasks this may be ~1× — think about *why* the cost gap grows with the number
   of tasks / retrain frequency, which is the real $100k story. Does this tiny
   setup actually capture H2, or do you need to vary task count to see it?
3. **How wrong were your predictions?** Quantify it. That gap is the learning.
4. Paste the `ENV` dict + `git rev-parse --short HEAD` into README 'Setup'.
5. Commit — including if results are boring or contradict you. Negative results
   are committed (methodology rule 4).

Then bring the numbers to the Socratic session before designing experiment 0002.